In [ ]:
%load_ext autoreload
%autoreload 2

from functools import partial

import moe
from tests import utils as test_utils

import jax
import jax.numpy as jnp
from jax.sharding import PartitionSpec as P
import tune_jax
import numpy as np

tune_jax.logger.setLevel("INFO")

In [ ]:
import contextlib
from pathlib import Path


@contextlib.contextmanager
def profile(path="/tmp/profiles", port=8791):
  with jax.profiler.trace("/tmp/profiles"):
    yield
  profiles = sorted(Path(path).absolute().glob("**/*.xplane.pb"), key=lambda x: x.stat().st_mtime)
  profile_name = profiles[-1].parts[-2]
  if port == 8791:
    url = "http://localhost:{port}/data/plugin/profile/trace_viewer@;run={name};tag=trace_viewer@"  # xprof version
  else:
    url = "http://localhost:{port}/?run={name}&tag=trace_viewer"  # tensorboard version
  print(url.format(port=port, name=profile_name))

In [ ]:
x = jax.random.normal(jax.random.key(0), (24, 1024))
idx = jax.random.randint(jax.random.key(0), (24,), minval=0, maxval=8)


def fn(x):
  return moe.utils.padded_group_gather(x, idx, max_idx=8, multiple=4)


dout = jnp.ones_like(x) * jnp.arange(x.shape[0])[:, None] + 7

In [ ]:
o, vjp_fn = jax.vjp(lambda *args: fn(*args)[0], x)
dout_ = dout[moe.utils.spread_arange_gather(o.shape[0], jnp.bincount(idx), 4)[0]][:, 0]
dout_ = moe.utils.padded_group_gather(dout, idx, max_idx=8, multiple=4)[0]
vjp_fn(dout_)[0][:, 0]

In [ ]:
jax.jvp(lambda x: x[jnp.argsort(idx), ...], (x,), (dout,))[1][:, 0]

In [ ]:
dout[:, 0]

In [ ]:
counts = jnp.bincount(idx, length=8)
padding_idxs = moe.utils.add_indices(jnp.arange(8), -counts % 4, max_size=4 - 1)
idx_with_padding = jnp.concat([idx, padding_idxs], axis=0)
gather_idx = jnp.argsort(idx_with_padding)

In [ ]:
dout_[:, 0][jnp.argsort(gather_idx)[jnp.argsort(idx)]]

In [ ]:
r = jax.random.normal(jax.random.key(0), (x.shape[0],))


def fn1(x, idx):
  y = x[jnp.argsort(idx), ...]
  print(r)
  print(y)
  return jnp.sum(y * r[:, None])


def fn2(x, idx):
  y = moe.utils.padded_group_gather(x, idx, max_idx=8, multiple=4)
  r_idx, mask = moe.utils.spread_arange_gather(y.shape[0], jnp.bincount(idx), multiple=4)
  r_ = r[r_idx, ...] * mask
  print(mask)
  print(r_)
  print(y)
  return jnp.sum(y * r_[:, None])



In [ ]:
y = fn(x)
y

In [ ]:
fn1(x, idx)

In [ ]:
fn2(x, idx)

In [ ]:
jax.jvp(fn, (x,), (dout,))[1][0][:, 0]

In [ ]:
m, k, n = (8192, 4096, 128)
s = jnp.ones((m, k), "bfloat16")
v = jnp.ones((k, n), "bfloat16")

fn = jax.jit(jnp.dot)

with profile():
  for _ in range(3):
    jax.block_until_ready(fn(s, v))
  for _ in range(3):
    jax.block_until_ready(fn(v.T, s.T))

In [ ]:
jnp.bincount(moe.utils.add_indices(jnp.array([1, 3, 5]), jnp.array([0, 7, 2]), 10), length=16)

In [ ]:
jnp.bincount(jnp.arange(12), length=2)

# actual moe testing

In [ ]:
n_devices = jax.device_count()
mesh = jax.make_mesh((n_devices,), ("x",), axis_types=(jax.sharding.AxisType.Explicit,))
jax.sharding.set_mesh(mesh)

In [ ]:
x, meta = test_utils.generate_data(4096, 7168, device_num=n_devices, axis_name="x")

In [ ]:
g = 32
all_idxs = jax.random.randint(jax.random.key(0), (2 * x.shape[0],), minval=0, maxval=g)
x2 = moe.core.run_moe(x, all_idxs, axis_name="x", experts_num=g, multiple=8)

In [ ]:
x2.shape

In [ ]:
x.shape

In [ ]:
jnp.sum(jnp.abs(jnp.sum(x2[:, :1, ...], axis=1) - x))

In [ ]:
x.at[1024 - 15:1024, 0, ...].get(out_sharding=P())

In [ ]:
x2.at[1024 - 15:1024, 0, 0, ...].get(out_sharding=P())

In [ ]:
jnp.cumsum(partial(jnp.bincount, length=g)(all_idxs), axis=-1)



In [ ]:
def print_me(i):
  print(x2.at[i - 1:i + 2, 0, 0, ...].get(out_sharding=P()))
  print(x.at[i - 1:i + 2, 0, ...].get(out_sharding=P()))



In [ ]:
print_me(868)

In [ ]:
print_me(4086)

In [ ]:
jnp.where(jnp.sum(jnp.abs(jnp.sum(x2[:, :1, ...], axis=1) - x), axis=-1).at[:, 0].get(out_sharding=P()))

In [ ]:
(1024 - 9) / 1024

In [ ]:
n_devices = jax.device_count()
axis_name = "x"
mesh = jax.make_mesh((n_devices,), (axis_name,), axis_types=(jax.sharding.AxisType.Explicit,))

experts_per_tok = 2

with jax.sharding.set_mesh(mesh):
  n, k = 16, 2048
  g = 32  # experts
  x, ra2a_meta = test_utils.generate_data(n, k, n_devices, axis_name="x")
  del ra2a_meta
  all_idxs = jax.random.randint(jax.random.key(0), (experts_per_tok * x.shape[0],), minval=0, maxval=g)
  out = moe.core.run_moe(x, all_idxs, axis_name="x", experts_num=n_devices)
  x_new = np.array(out[:, 0, :, :])
  np.testing.assert_allclose(x, x_new)

In [ ]:
all_idxs

In [ ]:
x

In [ ]:
jnp.sum(jnp.abs(x_new - x))

In [ ]:
x.s

In [ ]:
x

In [ ]:
x_new

# mesh experiments

In [ ]:
n = jax.device_count()
mesh = jax.make_mesh((n,), ("x",), axis_types=(jax.sharding.AxisType.Explicit,))
jax.sharding.set_mesh(mesh)

In [ ]:
fn = lambda: moe.utils.empty((1024, 1024), jnp.bfloat16, P(None, "x"))

In [ ]:
fn_ = tune_jax.tune(fn)
fn_()

# ra2a 2d

In [ ]:
n_devices = jax.device_count()
mesh = jax.make_mesh((n_devices,), ("x",), axis_types=(jax.sharding.AxisType.Explicit,))
jax.set_mesh(mesh)

In [ ]:
x_sort, input_offsets, send_sizes, output_offsets, recv_sizes = test_utils.generate_data(8 * 8 * 4096, 4096, n_devices, multiple=8)
x_sort = x_sort.reshape((x_sort.shape[0], -1)).astype(jnp.bfloat16)



In [ ]:
@jax.jit
@partial(jax.shard_map, out_specs=P("x", None), check_vma=False)
def test_kernel_async(x, input_offsets, send_sizes, output_offsets, recv_sizes):
  # output = jnp.zeros((2 * x.shape[0],) + x.shape[1:], x.dtype)
  output = moe.utils.empty((2 * x.shape[0],) + x.shape[1:], x.dtype)
  with jax.named_scope("start"):
    out = moe.ra2a.ra2a_2d(x, output, input_offsets, send_sizes, output_offsets, recv_sizes, axis_name="x", multiple=8)
  with jax.named_scope("jax.lax.ragged_all_to_all"):
    out2 = jax.lax.ragged_all_to_all(x, output, input_offsets, send_sizes, output_offsets, recv_sizes, axis_name="x")
  return out, out2



In [ ]:
out, out2 = test_kernel_async(x_sort, input_offsets, send_sizes, output_offsets, recv_sizes)
err = jnp.sum(jnp.abs(out - out2))
print(f"{err = }")

In [ ]:
with jax.profiler.trace("/tmp/ra2a"):
  for _ in range(3):
    out, out2 = jax.block_until_ready(test_kernel_async(x_sort, input_offsets, send_sizes, output_offsets, recv_sizes))

# compute on

In [ ]:
import jax
import jax.numpy as jnp
from jax.experimental.compute_on import compute_on

import numpy as np


@jax.jit
@compute_on("tpu_sparsecore")
def my_gather(x, idx):
  return x[idx, ...]


x = jnp.ones((8192, 4096))
idx = jnp.argsort(np.random.randn(x.shape[0]))

In [ ]:
with jax.profiler.trace("/tmp/compute_on"):
  for _ in range(3):
    jax.block_until_ready(my_gather(x, idx))